# 05 · Figures for the README

**Goal:** export the plots that illustrate the README as PNG files in `figures/`. Each figure is rebuilt here from the same code and data as the notebook it comes from, so the README images can be reproduced.

**Figures**
1. CGM and CBG for one patient (from `00_eda.ipynb`)
2. Features vs HbA1c (from `00_eda.ipynb`): the features are weak predictors
3. Measured vs predicted HbA1c for the best linear model (from `01_linear_model.ipynb`)
4. Measured vs predicted HbA1c for the best Random Forest (from `01_random_forest.ipynb`)
5. PCA of CGM profiles (from `02_dim_reduction.ipynb`)
6. GMI vs measured HbA1c (from `04_gmi_baseline.ipynb`)

**Best models** are chosen by the highest validation (CV) R² in their notebook: the linear model with TAR and TBR, and the Random Forest tuned by grid search on the 20 features.

## 1. Setup

In [1]:
# Make the project root importable so `src` can be imported from this notebook
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))
DATA_DIR = FILE / 'data' / 'processed' / 'data'
FIG_DIR = FILE / 'figures'
FIG_DIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error

from src.load_model_data import load_all_time_series, load_model_data, get_patient_hba1c_cgm
from src.prepare_splits import load_splits
from src.vis_utils import plot_patient_timeseries, feat_vs_target


def save_figure(fig, name, width, height):
    """Write a figure to figures/ as a PNG at twice the display resolution."""
    path = FIG_DIR / name
    fig.write_image(path, width=width, height=height, scale=2)
    print(f'saved {path.relative_to(FILE)}')


def gt_vs_pred(y_true, y_pred, title):
    """Measured vs predicted HbA1c with the perfect-prediction line."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y_true, y=y_pred, mode='markers',
                             marker=dict(size=9, opacity=0.7), name='test patients'))
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                             line=dict(dash='dash', color='grey'), name='perfect prediction'))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    fig.update_layout(
        title=f"{title}<br><sup>test R² = {r2_score(y_true, y_pred):.2f}, "
              f"RMSE = {rmse:.1f} mmol/mol</sup>",
        xaxis_title='Measured HbA1c (mmol/mol)',
        yaxis_title='Predicted HbA1c (mmol/mol)',
        height=500, width=500, legend=dict(orientation='h', y=-0.2)
    )
    return fig

## 2. CGM and CBG for one patient

The same plot as in `00_eda.ipynb`: CGM as a continuous line, CBG finger-stick readings as markers, and the glucose zones shaded behind them. This patient is the first T1DM patient in the summary file.

In [2]:
# CGM and CBG for the first T1DM patient, as in 00_eda.ipynb
all_ts = load_all_time_series(DATA_DIR)
patient_id = '1001_0_20210730'
fig = plot_patient_timeseries(all_ts[patient_id], patient_id,
                              include_cols=['CGM (mg / dl)', 'CBG (mg / dl)'])
fig.update_layout(title=f'CGM and CBG for patient {patient_id} (T1DM)', height=420, width=1000)
save_figure(fig, 'cgm_cbg_patient.png', width=1000, height=420)
fig

saved figures\cgm_cbg_patient.png


## 3. Features vs HbA1c

The scatter plots from `00_eda.ipynb`, with a linear fit and Pearson r in each axis label. Most panels show a flat or noisy cloud, which is the visual version of the weak correlations in the README.

In [3]:
# Features vs measured HbA1c for all 116 patients
model_data = load_model_data()
feature_cols = ['tar', 'tir', 'tbr', 'age', 'bmi', 'diabetes_duration',
                'cgm_mean', 'cgm_std', 'rapid_changes', 'hba1c']
fig = feat_vs_target(model_data[feature_cols], target_col='hba1c')
fig.update_layout(height=900, width=900, title='Features vs HbA1c (116 patients)')
save_figure(fig, 'features_vs_hba1c.png', width=900, height=900)
fig

saved figures\features_vs_hba1c.png


## 4. Measured vs predicted HbA1c: best linear model

The best linear model by validation R² is the one with TAR and TBR (`01_linear_model.ipynb`, experiment 1). It is trained on the 92 training patients and plotted on the 24 test patients. Points on the dashed line are perfect predictions.

In [4]:
# Best linear model: TAR and TBR, trained on the saved training split
splits = load_splits()
train, test = splits['train'], splits['test']
lin_cols = ['tar', 'tbr']
linear = LinearRegression().fit(train[lin_cols], train['hba1c'])
pred_linear = linear.predict(test[lin_cols])

fig = gt_vs_pred(test['hba1c'], pred_linear, 'Linear regression (TAR, TBR)')
save_figure(fig, 'linear_gt_vs_pred.png', width=500, height=500)
fig

saved figures\linear_gt_vs_pred.png


## 5. Measured vs predicted HbA1c: best Random Forest

The best forest by validation R² is the grid-searched model on the 20 features (`01_random_forest.ipynb`, experiment 4). The same search space and 5-fold CV on the training split are used here, so the model is the same as in that notebook.

In [5]:
# Best Random Forest: grid search on the 20-feature set, trained on the training split
rf_cols = ['tar', 'tbr', 'age', 'gender', 'bmi', 'diabetes_type', 'diabetes_duration',
           'cgm_mean', 'cgm_std', 'cgm_min', 'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75',
           'cgm_iqr', 'glucose_variability', 'n_hypo_events', 'n_hyper_events',
           'rapid_changes', 'mean_excursion']
param_grid = {
    'max_depth': [5, 8, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2'],
}
search = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid,
                      cv=5, scoring='r2').fit(train[rf_cols], train['hba1c'])
print('Best parameters:', search.best_params_)
print(f'Best CV R²: {search.best_score_:.4f}')
pred_rf = search.best_estimator_.predict(test[rf_cols])

fig = gt_vs_pred(test['hba1c'], pred_rf, 'Random Forest (grid search, 20 features)')
save_figure(fig, 'rf_gt_vs_pred.png', width=500, height=500)
fig

Best parameters: {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5}
Best CV R²: 0.2018


saved figures\rf_gt_vs_pred.png


## 6. PCA of CGM profiles

The same profiles as in `02_dim_reduction.ipynb`: 21 percentiles of each patient's CGM readings, standardized, then PCA. Points are colored by measured HbA1c. The left panel is PC1 vs PC2 and the right panel is PC1 vs PC3.

In [6]:
# PCA of per-patient CGM percentiles, colored by HbA1c
patient_cgm = get_patient_hba1c_cgm()
percentiles = np.arange(0, 101, 5)
patient_ids = list(patient_cgm.keys())
hba1c_vals = np.array([patient_cgm[p]['hba1c'] for p in patient_ids])
X = StandardScaler().fit_transform(
    np.array([np.percentile(patient_cgm[p]['cgm'], percentiles) for p in patient_ids]))
pca = PCA(n_components=3, random_state=42)
emb = pca.fit_transform(X)
var = pca.explained_variance_ratio_ * 100
print('Explained variance (%):', var.round(1))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
                    subplot_titles=('PC1 vs PC2', 'PC1 vs PC3'))
for col, j in [(1, 1), (2, 2)]:
    marker = dict(color=hba1c_vals, colorscale='Viridis', size=9, opacity=0.8,
                  showscale=(col == 2))
    if col == 2:
        marker['colorbar'] = dict(title='HbA1c (mmol/mol)')
    fig.add_trace(go.Scatter(x=emb[:, 0], y=emb[:, j], mode='markers', marker=marker,
                             text=patient_ids, showlegend=False,
                             hovertemplate='%{text}<br>HbA1c: %{marker.color:.1f}<extra></extra>'),
                  row=1, col=col)
fig.update_xaxes(title_text=f'PC1 ({var[0]:.0f}% of variance)', row=1, col=1)
fig.update_xaxes(title_text=f'PC1 ({var[0]:.0f}% of variance)', row=1, col=2)
fig.update_yaxes(title_text=f'PC2 ({var[1]:.0f}%)', row=1, col=1)
fig.update_yaxes(title_text=f'PC3 ({var[2]:.0f}%)', row=1, col=2)
fig.update_layout(title='PCA of CGM percentile profiles, colored by HbA1c',
                  height=420, width=950)
save_figure(fig, 'pca_components.png', width=950, height=420)
fig

Explained variance (%): [82.3 12.9  2.5]


saved figures\pca_components.png


## 7. GMI vs measured HbA1c

The same comparison as `04_gmi_baseline.ipynb`, on all 116 patients. GMI is converted from % to mmol/mol with the IFCC formula before plotting. Points below the dashed line are underestimates.

In [7]:
# Raw GMI in mmol/mol vs measured HbA1c, as in 04_gmi_baseline.ipynb
GMI_INTERCEPT, GMI_SLOPE = 3.31, 0.02392
IFCC_OFFSET, IFCC_FACTOR = 2.15, 10.929
gmi = IFCC_FACTOR * ((GMI_INTERCEPT + GMI_SLOPE * model_data['cgm_mean']) - IFCC_OFFSET)
bias = np.mean(gmi - model_data['hba1c'])
print(f'Bias of raw GMI: {bias:.1f} mmol/mol')

fig = gt_vs_pred(model_data['hba1c'], gmi, 'GMI (raw) vs measured HbA1c, all 116 patients')
fig.update_layout(xaxis_title='Measured HbA1c (mmol/mol)',
                  yaxis_title='GMI converted to mmol/mol')
save_figure(fig, 'gmi_vs_hba1c.png', width=600, height=500)
fig

Bias of raw GMI: -25.3 mmol/mol


saved figures\gmi_vs_hba1c.png
